[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Recurrent Networks &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import math
import os
from datetime import datetime, timedelta

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


oslo = np.array(year_of_readings("Oslo"), dtype="float32")
NOVEMBER = 24 * 304   # the hour 1 November begins, counted from midnight on 1 January

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(oslo), "hourly readings at Oslo")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 8760 hourly readings at Oslo


The worked examples' windows, tensors, forecaster and training loop, built from the same made-up
readings, rebuilt here so that each answer runs on its own.

In [2]:
windows = np.lib.stride_tricks.sliding_window_view(oslo, 25)
train_windows, test_windows = windows[:NOVEMBER - 24], windows[NOVEMBER:]
mean, std = float(oslo[:NOVEMBER].mean()), float(oslo[:NOVEMBER].std())


def as_tensors(rows, steps=24):
    scaled_rows = (rows - mean) / std
    return (torch.from_numpy(np.ascontiguousarray(scaled_rows[:, :steps])).unsqueeze(-1),
            torch.from_numpy(np.ascontiguousarray(scaled_rows[:, steps])).unsqueeze(-1))


X_train, y_train = as_tensors(train_windows)
X_test, y_test = as_tensors(test_windows)


class Forecaster(torch.nn.Module):
    def __init__(self, cell=torch.nn.LSTM, batch_first=True):
        super().__init__()
        self.rnn = cell(input_size=1, hidden_size=16, batch_first=batch_first)
        self.head = torch.nn.Linear(16, 1)

    def forward(self, x):
        output, _ = self.rnn(x)
        return self.head(output[:, -1])


def train(model, X_train, y_train, X_test, y_test, epochs=8):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    loss_fn = torch.nn.MSELoss()
    generator = torch.Generator().manual_seed(0)
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(X_train), generator=generator)
        for start in range(0, len(X_train), 64):
            rows = order[start:start + 64]
            optimizer.zero_grad()
            loss_fn(model(X_train[rows]), y_train[rows]).backward()
            optimizer.step()
    with torch.no_grad():
        return loss_fn(model(X_test), y_test).item() * std ** 2


torch.manual_seed(0)
lstm_forecaster = Forecaster()
lstm_test_mse = train(lstm_forecaster, X_train, y_train, X_test, y_test)

**1.** Make an `nn.LSTM` with `hidden_size=8`, `num_layers=2` and `batch_first=True`, give it the first 5 training windows, and print the shapes of its output and of its hidden state.

In [3]:
torch.manual_seed(0)
two_layers = torch.nn.LSTM(input_size=1, hidden_size=8, num_layers=2, batch_first=True)
output, (hidden, cell) = two_layers(X_train[:5])
print(tuple(output.shape), tuple(hidden.shape))

(5, 24, 8) (2, 5, 8)


The output is the top layer's state at every step, (5, 24, 8), and the hidden state has a row for each of the two layers, (2, 5, 8).

**2.** Print the square root of the baseline's test error and of `lstm_forecaster`'s, to read both in degrees.

In [4]:
baseline_mse = float(((test_windows[:, 23] - test_windows[:, 24]) ** 2).mean())
print("baseline", round(baseline_mse ** 0.5, 2), "degrees | LSTM", round(lstm_test_mse ** 0.5, 2), "degrees")

baseline 0.85 degrees | LSTM 0.65 degrees


The root mean square error is 0.85 degrees for the baseline and 0.65 for the LSTM, about 0.2 degrees less.

**3.** Cut the readings into windows of 6 hours and the hour after, train a `Forecaster` on them from `torch.manual_seed(0)`, and compare its test error with the 24-hour windows'.

In [5]:
short = np.lib.stride_tricks.sliding_window_view(oslo, 7)
X_short, y_short = as_tensors(short[:NOVEMBER - 6], steps=6)
X_short_test, y_short_test = as_tensors(short[NOVEMBER + 18:], steps=6)
print(len(X_short_test), "test windows")

torch.manual_seed(0)
short_forecaster = Forecaster()
print("6 hours:", round(train(short_forecaster, X_short, y_short, X_short_test, y_short_test), 3),
      "| 24 hours:", round(lstm_test_mse, 3))

1440 test windows
6 hours: 0.633 | 24 hours: 0.419


With 6 hours the forecaster's test error is 0.633, against 0.419 with 24, and still below the baseline's 0.729. `as_tensors` takes the number of steps, so the 7-reading windows split into 6 hours and the hour after. The test windows start 18 hours into November, so their forecast hours are the same 1,440 as the 24-hour windows'.

**4.** Build the Keras forecaster with a `GRU` layer in place of the `LSTM`, train it from `keras.utils.set_random_seed(0)`, and print its parameter count and test error.

In [6]:
keras.utils.set_random_seed(0)
gru_keras = keras.Sequential([
    keras.Input((24, 1)),
    keras.layers.GRU(16),
    keras.layers.Dense(1),
])
gru_keras.compile(optimizer=keras.optimizers.Adam(0.005), loss="mse")
gru_keras.fit(X_train.numpy(), y_train.numpy(), epochs=8, batch_size=64, verbose=0)
print(gru_keras.count_params(), "parameters")
print("test error", round(gru_keras.evaluate(X_test.numpy(), y_test.numpy(), verbose=0) * std ** 2, 3))

929 parameters
test error 0.595


The GRU forecaster has 929 parameters, the same as PyTorch's, and a test error of 0.595, above the Keras LSTM's 0.341 and below the baseline's 0.729. As the worked examples say, one run of each is not enough to rank them.

**5.** Forecast the first hour of 1 December, `oslo[24 * 334]`, from the 24 hours before it with `lstm_forecaster`, in degrees, and print it beside the reading and the baseline's forecast.

In [7]:
december = 24 * 334
window = torch.from_numpy((oslo[december - 24:december] - mean) / std).reshape(1, 24, 1)
with torch.no_grad():
    forecast = lstm_forecaster(window).item() * std + mean
print("forecast", round(forecast, 1), "| reading", round(float(oslo[december]), 1),
      "| last reading", round(float(oslo[december - 1]), 1))

forecast -2.4 | reading -1.7 | last reading -1.3


The forecaster says −2.4 degrees, where the reading was −1.7 and repeating the last reading would have said −1.3, so for this hour the baseline was closer. The test error is an average over 1,440 hours, and a single hour can go either way.

---

&#8592; **Back to:** [Recurrent Networks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/13-recurrent-networks.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
